In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from absl import app, flags, logging
from ml_collections import config_flags
import os
from tqdm.notebook import trange

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

import matplotlib.patches as patches
from scipy.optimize import curve_fit
import numpy.linalg as la

import plotly.io as pio
pio.renderers.default = "notebook_connected"

import json
from ml_collections import ConfigDict

import plotly.graph_objects as go

from icl.linear.train_linear import train
from icl.linear.lr_config import get_config
from icl.linear.lr_task import *
from icl.linear.linear_utils import *
from icl.linear.train_linear import get_sharded_batch_sampler
from icl.linear import DiscreteMMSE, Ridge
from icl.utils import visualize_attention
from icl.figures.task_vec_viz import *

logging.set_verbosity(logging.INFO)
torch.set_printoptions(precision=3, sci_mode=False)
np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

In [2]:
# View attention map
def view_attn(train_task):
    train_task.batch_size = 1
    demo_data0, demo_target = train_task.sample_from_task(train_task.task_pool[0], step=2)
    attns = get_attn(model, demo_data0, demo_target)
    cap = 100
    attns_capped = {layer_key: tensor[:, :cap, :cap] for layer_key, tensor in attns.items()}
    
    widget = visualize_attention(attns_capped, mode='widget')
    return widget

# Utility function to load model and task sampler
def load_model_and_task(exp_name):
    work_dir = os.path.join("..", "results", "linear")
    exp_dir = os.path.join(work_dir, exp_name)
    config_path = os.path.join(exp_dir, "config.json")
    with open(config_path, "r") as f:
        config_dict = json.load(f)
    
    config = ConfigDict(config_dict)
    checkpoint_path = os.path.join(exp_dir, "checkpoint.pt")
    checkpoint = torch.load(checkpoint_path, map_location=config.device)
    data_type = torch.float
    model = get_model(**config["model"], dtype=data_type)
    model.load_state_dict(checkpoint["model"])
    train_task = get_task(**config["task"])
    return model, train_task

def get_attn_mean_var(train_task, model):
    train_task = get_task(**config["task"])
    train_task.batch_size = 256
    demo_data, demo_target = train_task.sample_from_task(train_task.task_pool[1], step=2)
    attns = get_attn(model, demo_data, demo_target)
    attn_means = {layer_key: tensor[:, :, 1::3].mean(dim=0).norm(dim=(-1,-2)).square().cpu().item() for layer_key, tensor in attns.items()}
    attn_vars = {layer_key: tensor[:, :, 1::3].var(dim=0).sum(dim=(-1,-2)).cpu().item() for layer_key, tensor in attns.items()}
    return attn_vars, attn_means

def check_injection(train_task, model, inject_vectors, layer=1, pos=1, is_diff=False, task_idx=None):
    t0 = pos
    l0 = layer
    train_task.batch_size = 1024
    # print(f"Memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    criterion = nn.MSELoss(reduction='none')
    # tvs_means = task_vectors.mean(dim=-2)
    # tvs_mean_weighted = tvs_means[:, -10:].mean(dim=1)
    
    for k in range(config.task.n_tasks):
        torch.cuda.empty_cache()
        if task_idx is None:
            tid = k
        else:
            tid = task_idx
        
        query_data, query_target = train_task.sample_from_task(train_task.task_pool[tid], step=50)
        
        preds_rand = predict_with_task_vector(
                model=model,
                query_data=query_data[:, :(3*t0+3)],
                query_target=query_target[:, :(3*t0+3)],
                task_vector=inject_vectors[k],
                l=l0,              # same layer
                pad="mapsto",      # same position
                pos=3*t0+1,
                is_diff=is_diff
            )
        with torch.no_grad():
            preds = model(query_data, query_target)

        if is_diff:
            query_target[:, t0] = (query_data[:, t0] @ train_task.task_pool[k]).squeeze(-1)
        
        errors_rand = criterion(preds_rand[:, t0], query_target[:, t0].to(preds_rand.device))  # shape: (batch,)
        errors_base = criterion(preds[:, t0], query_target[:, t0].to(preds_rand.device))
        loss = errors_rand.mean()
        baseline_loss = errors_base.mean()
        
        std_rand = errors_rand.std(unbiased=True)
        std_base = errors_base.std(unbiased=True)
        print(f"{t0}-shot loss w. injected vector: {loss.item():.3f} ({std_rand.item():.3f})")
        print(f"{t0}-shot loss w.o. injected vector: {baseline_loss.item():.3f} ({std_base.item():.3f})")

def batch_cosine_similarity(A: np.ndarray, B: np.ndarray, eps: float = 1e-8):
    dot_product = np.sum(A * B, axis=1)
    norm_A = np.linalg.norm(A, axis=1)
    norm_B = np.linalg.norm(B, axis=1)
    return dot_product / (norm_A * norm_B + eps)

def rolling_mean_l2_deviation(arr: torch.Tensor, window: int):
    B, T, D = arr.shape
    output = torch.empty(B, T - window, device=arr.device)

    for b in range(B):
        for t in range(T - window):
            window_slice = arr[b, t:t+window, :]  # (W, D)
            window_mean = window_slice.mean(dim=0)  # (D,)
            l2_distances = torch.norm(window_slice - window_mean, dim=1)  # (W,)
            output[b, t] = l2_distances.mean()

    return output


def plot_mse_vs_position(model, samplers_eval, bayes_ood, bayes_id, step: int = 1):
    import numpy as np
    import torch
    import plotly.graph_objects as go

    torch.cuda.empty_cache()

    def compute_metrics(mode: str):
        _data, _, _target = samplers_eval[mode](step=step)
        _data = _data.squeeze(0)
        _target = _target.squeeze(0)

        with torch.no_grad():
            preds = model(_data, _target)
        loss = ((preds - _target.to(preds.device))**2).mean(dim=0)

        re_preds = bayes_ood(_data, _target)
        re_loss = ((re_preds - _target.to(re_preds.device))**2).mean(dim=0)

        dmmse_preds = bayes_id(_data, _target)
        dmmse_loss = ((dmmse_preds - _target.to(dmmse_preds.device))**2).mean(dim=0)

        diff_dmmse = np.abs(preds.mean(dim=0).cpu().numpy() - dmmse_preds.mean(dim=0).cpu().numpy())
        diff_re = np.abs(preds.mean(dim=0).cpu().numpy() - re_preds.mean(dim=0).cpu().numpy())

        return (
            loss.cpu().numpy(),
            re_loss.cpu().numpy(),
            dmmse_loss.cpu().numpy(),
            diff_dmmse,
            diff_re,
        )

    results = {
        "ID": compute_metrics("Pretrain"),
        "OOD": compute_metrics("Latent"),
    }

    t = np.arange(1, results["ID"][0].shape[0] + 1)

    fig = go.Figure()
    modes = ["ID", "OOD"]
    views = ["MSE", "Δ"]
    trace_labels = {
        "MSE": ["Model MSE", "Ridge MSE", "dMMSE MSE"],
        "Δ": ["dMMSE Δ", "Ridge Δ"]
    }
    line_styles = [
        dict(width=2),
        dict(width=2, dash="dash"),
        dict(width=2, dash="dot")
    ]

    # Add all traces
    for mode in modes:
        vals = results[mode]
        for i in range(3):
            fig.add_trace(go.Scatter(
                x=t, y=vals[i], mode="lines+markers",
                name=f"{trace_labels['MSE'][i]} ({mode})",
                line=line_styles[i],
                marker=dict(size=4),
                visible=(mode == "ID" and i < 3)  # default
            ))
        for i in range(3, 5):
            fig.add_trace(go.Scatter(
                x=t, y=vals[i], mode="lines+markers",
                name=f"{trace_labels['Δ'][i - 3]} ({mode})",
                line=line_styles[i - 3],
                marker=dict(size=4),
                visible=False
            ))

    # Utility: visibility mask for 4 modes
    def vis_mask(mode, view):
        out = []
        for m in modes:
            for i in range(5):
                out.append((m == mode and ((view == "MSE" and i < 3) or (view == "Δ" and i >= 3))))
        return out

    # One dropdown, 4 buttons
    dropdown = dict(
        buttons=[
            dict(label="MSE vs Position (ID)",
                 method="update",
                 args=[
                     {"visible": vis_mask("ID", "MSE")},
                     {"title": {"text": "MSE vs Position (ID)"},
                      "yaxis": {"title": "MSE"}}
                 ]),
            dict(label="Prediction Difference (ID)",
                 method="update",
                 args=[
                     {"visible": vis_mask("ID", "Δ")},
                     {"title": {"text": "Prediction Difference (ID)"},
                      "yaxis": {"title": "Absolute Difference"}}
                 ]),
            dict(label="MSE vs Position (OOD)",
                 method="update",
                 args=[
                     {"visible": vis_mask("OOD", "MSE")},
                     {"title": {"text": "MSE vs Position (OOD)"},
                      "yaxis": {"title": "MSE"}}
                 ]),
            dict(label="Prediction Difference (OOD)",
                 method="update",
                 args=[
                     {"visible": vis_mask("OOD", "Δ")},
                     {"title": {"text": "Prediction Difference (OOD)"},
                      "yaxis": {"title": "Absolute Difference"}}
                 ]),
        ],
        direction="down",
        x=0.01,
        y=1.15,
        showactive=True,
        xanchor="left"
    )

    fig.update_layout(
        updatemenus=[dropdown],
        title="MSE vs Position (ID)",
        xaxis_title="Position",
        yaxis_title="MSE",
        template="plotly_white",
        height=500,
        width=800,
        legend=dict(title="Legend", itemsizing="constant")
    )
    fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='LightGray')

    fig.show()


def plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior):
    """
    Plots relative error curves for all k:
        rel_error_k(t) = ||approx_vec - true_vec|| / ||true_vec||
    where approx_vec = E_posterior[lambda] @ task_vectors[:-1, -1]

    Args:
        train_task: input to get_dmmse_posterior (custom format)
        task_vectors: Tensor of shape (num_tasks, seq_len, d)
        get_dmmse_posterior: function(train_task, k) -> (posterior, xs)
    """
    num_tasks, seq_len, _ = task_vectors.shape

    fig = go.Figure()
    x_vals = list(range(seq_len))

    for k in range(num_tasks):
        posterior, xs = get_dmmse_posterior(train_task, k)  # shape: (num_samples, num_tasks)
        bayes_lambdas = posterior.mean(dim=0)               # shape: (num_tasks,)
        approx_vecs = bayes_lambdas[:-1] @ task_vectors[:, -1]  # shape: (d,)
        true_vecs = task_vectors[k]                         # shape: (seq_len, d)

        # Relative error at each t
        rel_error = (approx_vecs - true_vecs).norm(dim=-1) / true_vecs.norm(dim=-1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=rel_error.cpu().numpy(),
            mode='lines+markers',
            name=f"k = {k}",
            hovertemplate="t: %{x}<br>Rel. error: %{y:.4f}<extra>k = " + str(k) + "</extra>"
        ))

    fig.update_layout(
        title="Relative Error for Posterior-Weighted Task Vector",
        xaxis_title="t (position)",
        yaxis_title="Relative Error",
        height=500,
        legend_title="Task Index k"
    )

    fig.show()

def plot_all_relative_errors_eval(eval_task, train_task, final_task_vectors, eval_vectors, get_dmmse_posterior_eval):
    """
    Plots relative error curves for all k:
        rel_error_k(t) = ||approx_vec - true_vec|| / ||true_vec||
    where approx_vec = E_posterior[lambda] @ final_task_vectors[:-1]

    Args:
        train_task: input to get_dmmse_posterior (custom format)
        task_vectors: Tensor of shape (num_tasks, seq_len, d)
        get_dmmse_posterior: function(train_task, k) -> (posterior, xs)
    """
    num_tasks, seq_len, _ = eval_vectors.shape

    fig = go.Figure()
    x_vals = list(range(seq_len))

    for k in range(num_tasks):
        posterior, xs = get_dmmse_posterior_eval(eval_task, train_task, k)  # shape: (num_samples, num_tasks)
        bayes_lambdas = posterior.mean(dim=0)               # shape: (num_tasks,)
        approx_vecs = bayes_lambdas[:-1] @ final_task_vectors  # shape: (d,)
        true_vecs = eval_vectors[k]                         # shape: (seq_len, d)

        # Relative error at each t
        rel_error = (approx_vecs - true_vecs).norm(dim=-1) / true_vecs.norm(dim=-1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=rel_error.cpu().numpy(),
            mode='lines+markers',
            name=f"k = {k}",
            hovertemplate="t: %{x}<br>Rel. error: %{y:.4f}<extra>k = " + str(k) + "</extra>"
        ))

    fig.update_layout(
        title="Relative Error for Posterior-Weighted Task Vector",
        xaxis_title="t (position)",
        yaxis_title="Relative Error",
        height=500,
        legend_title="Task Index k"
    )

    fig.show()

In [12]:
from sklearn.decomposition import PCA

def evaluate_and_estimate_lambdas(
    model,
    train_task,
    task_vectors,
    global_mean,
    config,
    K=3000,
    layer_index=3,
    weight_seed=None,
    weight_scale=2.0,
):
    """
    Generate evaluation tasks as linear combinations of anchor tasks, compute task vectors, 
    and estimate lambda weights and R^2 scores.

    Args:
        model: The neural model used for computing hidden representations.
        train_task: Task object containing `task_pool` (shape: [3, d, 1] or [3, d]).
        task_vectors: Tensor of shape (num_tasks, num_layers, d) containing anchor task vectors.
        global_mean: Tensor of shape (d,) to center hidden representations.
        config: Hydra config object or dict containing task configuration.
        K (int): Number of evaluation tasks.
        layer_index (int): Layer index to extract representations from.
        weight_seed (int or None): Optional random seed for reproducibility.

    Returns:
        lambdas (Tensor): shape (K, 3), estimated λ per evaluation task.
        r2_scores (Tensor): shape (K,), R² score of each fit.
        eval_task_vectors (Tensor): shape (K, d), representation of each evaluation task.
        eval_task (Task): task object containing the evaluation task pool.
    """
    if weight_seed is not None: torch.manual_seed(weight_seed)

    d = config.task.n_dims

    # Sample convex weights (not normalized) for eval task pool
    weights = weight_scale * torch.randn(K, 3)  # shape (K, 3)

    # Clone config and prepare eval task
    eval_config = config.copy() if isinstance(config, dict) else config
    eval_config.task.n_tasks = K
    eval_task = get_task(**eval_config["task"])

    # Create eval task pool: linear combinations of anchor tasks
    anchor_pool = train_task.task_pool.squeeze(-1)  # shape (3, d)
    eval_task_pool = weights @ anchor_pool  # shape (K, d)
    eval_task.task_pool = eval_task_pool.unsqueeze(-1)  # shape (K, d, 1)

    # Compute eval task vectors
    eval_hiddens, eval_xdata = compute_task_vectors(eval_config, model, eval_task, layer_index=layer_index)
    eval_task_vectors = eval_hiddens - global_mean.unsqueeze(0).unsqueeze(2)  # center
    eval_task_vectors = eval_task_vectors.mean(dim=-2)  # shape (K, d)

    # Estimate lambdas and R²
    lambdas, r2_scores = estimate_lambda_with_r2_fast(task_vectors[:, -1], eval_task_vectors)

    return lambdas, r2_scores, eval_task_vectors, eval_task, weights

def plot_lambda_projection(train_task_pool, eval_task_pool, lambdas, weights, title=None):
    """
    Plot convex combination projections of eval tasks onto anchor tasks using PCA and color coding for λ.

    Args:
        train_task_pool (Tensor): shape (3, d, 1) or (3, d) — anchor vectors
        eval_task_pool (Tensor): shape (K, d, 1) or (K, d) — evaluation vectors (not used for PCA)
        lambdas (Tensor or ndarray): shape (K, 3) — convex weights
        weights (Tensor or ndarray): shape (K, 3) — convex weights used for projection (can be same as lambdas)
        results (list or ndarray, optional): MSEs or other metrics to display per point
        title (str, optional): plot title
    """
    def get_mse_last(model, eval_task, train_task, step: int = 1):
        def compute_metrics(k):
            _data, _target = eval_task.sample_from_task(eval_task.task_pool[k], step=step)
            last_data = _data[:,-1] # (batch, n_dims)
            tasks = train_task.task_pool.squeeze(-1) # (n_tasks, n_dims)
            oracle_targets = (last_data @ tasks.transpose(0,1)).squeeze(-1)  # (batch, n_task)
    
            with torch.no_grad():
                preds = model(_data, _target) # (batch, n_points)
            loss = ((preds[:,-1:] - oracle_targets.to(preds.device))**2).mean(dim=0) # n_task
    
            return loss.cpu().numpy()
    
        results = np.zeros((len(eval_task.task_pool), 
                            len(train_task.task_pool)))
        
        for k in trange(len(eval_task.task_pool)):
            results[k] = compute_metrics(k) 
    
        return results
    
    results = get_mse_last(model, eval_task, train_task)
    # Step 1: Convert to NumPy
    anchor_np = train_task_pool.squeeze(-1).cpu().numpy() if torch.is_tensor(train_task_pool) else np.asarray(train_task_pool)
    eval_np = eval_task_pool.squeeze(-1).cpu().numpy() if torch.is_tensor(eval_task_pool) else np.asarray(eval_task_pool)
    lambda_np = lambdas.cpu().numpy() if torch.is_tensor(lambdas) else np.asarray(lambdas)
    weights_np = weights.cpu().numpy() if torch.is_tensor(weights) else np.asarray(weights)

    # Step 2: PCA on anchor points
    pca = PCA(n_components=2)
    anchor_2d = pca.fit_transform(anchor_np)

    # Step 3: Project eval points using λ ⋅ anchor_2d
    eval_2d = weights_np @ anchor_2d

    # Step 4: Color map from λ to RGB
    colors = lambda_np @ np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255]])
    colors = np.clip(colors, 0, 255).astype(int)
    colors_hex = [f"rgb({r},{g},{b})" for r, g, b in colors]

    # Step 5: Plot
    fig = go.Figure()

    # Eval points
    if results is not None:
        hover_text = [f"λ = {np.round(lambda_np[k], 2)}<br>MSE: {np.round(results[k], 2)}" for k in range(len(lambda_np))]
    else:
        hover_text = [f"λ = {np.round(lambda_np[k], 2)}" for k in range(len(lambda_np))]

    fig.add_trace(go.Scatter(
        x=eval_2d[:, 0],
        y=eval_2d[:, 1],
        mode='markers',
        marker=dict(size=6, color=colors_hex, opacity=0.8),
        name="Eval Points",
        hoverinfo='text',
        text=hover_text
    ))

    # Anchor points
    fig.add_trace(go.Scatter(
        x=anchor_2d[:, 0],
        y=anchor_2d[:, 1],
        mode='markers+text',
        marker=dict(size=10, color='black', symbol='x'),
        text=[f"w{i}" for i in range(anchor_2d.shape[0])],
        textposition='top center',
        name="Anchor Points"
    ))

    fig.update_layout(
        title=title or "Attraction to Anchor Points (λ0→R, λ1→G, λ2→B)",
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        width=700,
        height=700,
        showlegend=True
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()


### $p_{\text{minor}}=0$

In [6]:
# A lazy way to get results from a certain configuration.
# Most experiments share the same hyper parameters except `n_task`, so just change `n_task` and try train it
# If it is already trained then it will show where the results are saved.
# Otherwise, it will run the training procedure.
# This is a very ad hoc approach, as random seeds are generated differently on different systems and machines
# Need a fix in the future

config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 3
config.task.p_minor = 0
model, log = train(config)

..\results\linear\train_135d77ed8fb676a8c7be3c4dd84625ae


eval/Latent/Ridge,▁
eval/Latent/True,▁
eval/Latent/dMMSE,▁
eval/Pretrain/Ridge,▁
eval/Pretrain/True,▁
eval/Pretrain/dMMSE,▁
train/lr,▁█
eval/Latent/Ridge,1.04046
eval/Latent/True,1.08058
eval/Latent/dMMSE,1.14527
eval/Pretrain/Ridge,1.15977


KeyboardInterrupt: 

In [7]:
model, train_task = load_model_and_task("train_cdabfa7b05953c1da71ee0e8c5334f18")
model = model.to(config.device)
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

  0%|          | 0/3 [00:00<?, ?it/s]

In [134]:
task_idx = 0
inject_vectors = task_vectors[:,-1] - task_vectors[task_idx:(task_idx+1), -1]
check_injection(train_task, model, inject_vectors, layer=3, pos=20, is_diff=True, task_idx=task_idx)

20-shot loss w. injected vector: 0.000 (0.001)
20-shot loss w.o. injected vector: 0.000 (0.001)
20-shot loss w. injected vector: 0.307 (0.552)
20-shot loss w.o. injected vector: 14.920 (21.509)
20-shot loss w. injected vector: 0.541 (0.808)
20-shot loss w.o. injected vector: 18.174 (25.544)


In [135]:
def pairwise_cosine_similarity(X):
    X_norm = F.normalize(X, p=2, dim=1)  # Normalize each row to unit norm
    sim_matrix = X_norm @ X_norm.T       # Dot product between rows
    return sim_matrix
    
pairwise_cosine_similarity(task_vectors[:,-1]), pairwise_cosine_similarity(train_task.task_pool.squeeze(-1))

(tensor([[ 1.000, -0.549, -0.572],
         [-0.549,  1.000, -0.372],
         [-0.572, -0.372,  1.000]]),
 tensor([[ 1.000, -0.164, -0.085],
         [-0.164,  1.000,  0.380],
         [-0.085,  0.380,  1.000]]))

In [9]:
plot_task_vector_modes(hiddens)

In [171]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE)

In [6]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [173]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

In [10]:
lambdas, r2_scores, eval_task_vectors, eval_task, weights = evaluate_and_estimate_lambdas(model, train_task, 
                                                                                          task_vectors, global_mean, 
                                                                                          config, K=3000, layer_index=3)

  0%|          | 0/3000 [00:00<?, ?it/s]

In [11]:
plot_lambda_projection(train_task.task_pool, eval_task.task_pool, lambdas, weights)

  0%|          | 0/3000 [00:00<?, ?it/s]

In [ ]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE)

### $M=2^k$

In [121]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 3
for k in trange(2, 11, 2):
    config.task.n_minor_tasks = 2**k
    model, log = train(config)

  0%|          | 0/5 [00:00<?, ?it/s]

train_b4d9beea445f1d49fc1697c0e966bf41 already completed
Loaded model from ../results/linear/train_b4d9beea445f1d49fc1697c0e966bf41/checkpoint.pt
train_8abc8879de6b5cdda1264c3a970503fb already completed
Loaded model from ../results/linear/train_8abc8879de6b5cdda1264c3a970503fb/checkpoint.pt
train_86f6a7d2e94c32e2dd4392f8e678767c already completed
Loaded model from ../results/linear/train_86f6a7d2e94c32e2dd4392f8e678767c/checkpoint.pt
train_2e5842eb2a962e8009d51acbcd0b1d4e already completed
Loaded model from ../results/linear/train_2e5842eb2a962e8009d51acbcd0b1d4e/checkpoint.pt
train_2858ad40e5052c5197ab2d88702edbc3 already completed
Loaded model from ../results/linear/train_2858ad40e5052c5197ab2d88702edbc3/checkpoint.pt


In [13]:
model, train_task = load_model_and_task("train_2858ad40e5052c5197ab2d88702edbc3")
model = model.to(config.device)
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

  0%|          | 0/3 [00:00<?, ?it/s]

In [14]:
plot_task_vector_modes(hiddens)

In [15]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE)

In [81]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [131]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

In [16]:
lambdas, r2_scores, eval_task_vectors, eval_task, weights = evaluate_and_estimate_lambdas(model, train_task, 
                                                                                          task_vectors, global_mean, 
                                                                                          config, K=1000, layer_index=3, 
                                                                                          weight_scale=0.5)
plot_lambda_projection(train_task.task_pool, eval_task.task_pool, lambdas, weights)

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [103]:
import plotly.colors as pc

def sample_unit_vectors(n, d):
    x = torch.randn(n, d)           
    x = x / x.norm(dim=1, keepdim=True) 
    return x

K = 5
d = config.task.n_dims

eval_config = config.copy() if isinstance(config, dict) else config
eval_config.task.n_tasks = K
eval_task = get_task(**eval_config["task"])

eval_task_pool = train_task.task_pool[:1].squeeze(-1) + 0.5 * sample_unit_vectors(K,d) # torch.randn((K, d))  # shape (K, d)
groups = torch.zeros((K,))
eval_task_pool = torch.cat([eval_task_pool, 
                            train_task.task_pool[:1].squeeze(-1) + 2 * sample_unit_vectors(K,d)], dim=0)
groups = torch.cat([groups, torch.ones((K,))], dim=0)
eval_task_pool = torch.cat([eval_task_pool, 
                            train_task.task_pool[:1].squeeze(-1) + 1.2 * sample_unit_vectors(K,d)], dim=0)
groups = torch.cat([groups, 2*torch.ones((K,))], dim=0)
eval_task_pool = torch.cat([eval_task_pool, train_task.task_pool[:2].squeeze(-1)], dim=0)
groups = torch.cat([groups, torch.tensor([3,4])], dim=0)
eval_task.task_pool = eval_task_pool.unsqueeze(-1)  # shape (K, d, 1)
eval_hiddens, eval_xdata = compute_task_vectors(eval_config, model, eval_task, layer_index=3)
eval_task_vectors = eval_hiddens - global_mean.unsqueeze(0).unsqueeze(2)  # center
eval_task_vectors = eval_task_vectors.mean(dim=-2)  # shape (K, T, d)

groups = groups.long()
unique_groups = sorted(set(groups.tolist()))
group_to_color = {g: pc.qualitative.Set2[i % len(pc.qualitative.Set2)] for i, g in enumerate(unique_groups)}


ys = (eval_task_vectors - eval_task_vectors[-2:-1, -1:]).norm(dim=-1)
x = np.arange(1, ys.shape[1] + 1)
fig = go.Figure()
for k in range(ys.shape[0]):
    group_label = groups[k].item()
    fig.add_trace(go.Scatter(
        x=x,
        y=ys[k],
        mode='lines',
        name=str(group_label),
        opacity=1,
        line=dict(width=1, color=group_to_color[group_label])
    ))
fig.update_layout(
    xaxis_title="Position",
    width=800,
    height=500,
    template="plotly_white"
)
fig.show()

  0%|          | 0/17 [00:00<?, ?it/s]

In [104]:
A = eval_task_vectors.view(-1, 128)  
# Compute projection matrix P = Bᵗ (B Bᵗ)⁻¹ B
B = eval_task_vectors[-2:, -1:].squeeze(1).clone()  # (128, 2)
proj = A @ (B.T @ torch.inverse(B @ B.T) @ B)                   
residual = A - proj

# Reshape back
residual = residual.view(eval_task_vectors.shape)

ys = residual.norm(dim=-1)
x = np.arange(1, ys.shape[1] + 1)
fig = go.Figure()
for k in range(ys.shape[0]):
    group_label = groups[k].item()
    fig.add_trace(go.Scatter(
        x=x,
        y=ys[k],
        mode='lines',
        name=str(group_label),
        opacity=1,
        line=dict(width=1, color=group_to_color[group_label])
    ))
fig.update_layout(
    xaxis_title="Position",
    width=800,
    height=500,
    template="plotly_white"
)
fig.show()

In [ ]:
ys = (eval_task_vectors - eval_task_vectors[-2:-1, -1:]).norm(dim=-1)
x = np.arange(1, ys.shape[1] + 1)
fig = go.Figure()
for k in range(ys.shape[0]):
    fig.add_trace(go.Scatter(
        x=x,
        y=ys[k],
        mode='lines',
        name=k,
        opacity=1,
        line=dict(width=1)
    ))
fig.update_layout(
    xaxis_title="Position",
    width=800,
    height=500,
    template="plotly_white"
)
fig.show()

In [93]:
groups.long()

tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 3])

### $p=0.05$

In [184]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 4
config.task.p_ood = 0.05
model, log = train(config)

train_43e3e3efdc38445b8b255571cb502ac8 already completed
Loaded model from ../results/linear/train_43e3e3efdc38445b8b255571cb502ac8/checkpoint.pt


In [185]:
model, train_task = load_model_and_task("train_43e3e3efdc38445b8b255571cb502ac8")
model = model.to(config.device)
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)

  0%|          | 0/4 [00:00<?, ?it/s]

In [186]:
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

pairwise_cosine_similarity(task_vectors[:,-1])

tensor([[ 1.000, -0.060, -0.464, -0.214],
        [-0.060,  1.000, -0.502, -0.284],
        [-0.464, -0.502,  1.000, -0.403],
        [-0.214, -0.284, -0.403,  1.000]])

In [155]:
plot_task_vector_variance_with_fit(hiddens, normalize=False)

In [156]:
tvs_diff_means = plot_task_vector_differences(hiddens)

In [157]:
plot_pairwise_task_vector_variance(hiddens)

In [158]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [159]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [160]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [162]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

### $p=0.1$

In [76]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 3
config.task.p_minor = 0.1
model, log = train(config)

train_21928aaf4cb95072c39ab1cb9c56bc40 already completed
Loaded model from ../results/linear/train_21928aaf4cb95072c39ab1cb9c56bc40/checkpoint.pt


In [78]:
model, train_task = load_model_and_task("train_21928aaf4cb95072c39ab1cb9c56bc40")
model = model.to(config.device)
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)

global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

pairwise_cosine_similarity(task_vectors[:,-1])

  0%|          | 0/3 [00:00<?, ?it/s]

tensor([[ 1.000, -0.502, -0.662],
        [-0.502,  1.000, -0.315],
        [-0.662, -0.315,  1.000]])

In [79]:
plot_task_vector_variance_with_fit(hiddens, normalize=False)

In [80]:
tvs_diff_means = plot_task_vector_differences(hiddens)

In [81]:
plot_pairwise_task_vector_variance(hiddens)

In [82]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [83]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [84]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [89]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

In [90]:
eval_config = config
eval_config.task.n_tasks = 20
eval_task = get_task(**eval_config["task"])
eval_hiddens, eval_xdata = compute_task_vectors(eval_config, model, eval_task, layer_index=3)
eval_global_mean = eval_hiddens.mean(dim=(0,2))
eval_task_vectors = eval_hiddens - eval_global_mean.unsqueeze(0).unsqueeze(2)
eval_task_vectors = eval_task_vectors.mean(dim=-2)

lambdas, r2_scores = estimate_lambda_with_r2(normalize(task_vectors[:,-1]), normalize(eval_task_vectors))
plot_lambdas(lambdas)

  0%|          | 0/20 [00:00<?, ?it/s]

### Other settings

In [89]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**7
model, log = train(config)

train_bc1f7727fc8c8c5c6a412115ae4d24c5 already completed
Loaded model from ../results/linear/train_bc1f7727fc8c8c5c6a412115ae4d24c5/checkpoint.pt


In [97]:
model, train_task = load_model_and_task("train_bc1f7727fc8c8c5c6a412115ae4d24c5")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/128 [00:00<?, ?it/s]

In [98]:
plot_task_vector_variance_with_fit(task_vectors)

In [92]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [93]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/128 [00:00<?, ?it/s]

In [96]:
plot_task_vector_variance_with_fit(task_vectors)

In [95]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [101]:
plot_pairwise_task_vector_variance(task_vectors)

In [99]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [100]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [102]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [104]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [1642.259    0.519    0.041    0.017    0.01     0.006    0.006    0.004
    0.003    0.002    0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.      -0.      -0.      -0.
    0.       0.       0.      -0.      -0.      -0.      -0.      -0.
   -0.       0.       0.       0.       0.       0.      -0.      -0.
    0.       0.      -0.      -0.       0.       0.       0.      -0.
   -0.      -0.       0.      -0.      -0.       0.       0.       0.
   -0.      -0.       0.      -0.      -0.       0.      -0.       0.
   -0.       0.       0.       0.      -0.      -0.      -0.      -0.
    0.       0.      -0.       0.      -0.      -0.      -0.       0.
    0.      -0.      -0.       0.       0.       0.      -0.      -0.
   -0.      -0.      -0.       0.       0.       0.      -0.       0.
   -0.      -0.      -0.       0.       0.       0.       0.      -0.
   -0.      -0.      -0.      -0.       0.      -0.      -0.     

### 2**8

In [105]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**8
model, log = train(config)

train_21add59d65446de57febaab2b406b583 already completed
Loaded model from ../results/linear/train_21add59d65446de57febaab2b406b583/checkpoint.pt


In [116]:
model, train_task = load_model_and_task("train_21add59d65446de57febaab2b406b583")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/256 [00:00<?, ?it/s]

In [107]:
plot_task_vector_variance_with_fit(task_vectors)

In [108]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [117]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

/mnt/c/Users/User/LLM/ICL/.venv/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.196e-03, tolerance: 1.829e-03



In [118]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [4488.731   21.52     0.332    0.013    0.009    0.006    0.005    0.004
    0.004    0.003    0.001    0.       0.       0.       0.      -0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.       0.       0.       0.       0.
    0.       0.       0.       0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.       0.       0.       0.       0.
    0.      -0.      -0.      -0.      -0.      -0.      -0.       0.
    0.       0.       0.       0.       0.       0.       0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.      -0.      -0.       0.       0.
    0.      -0.      -0.      -0.      -0.       0.       0.     

In [110]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/256 [00:00<?, ?it/s]

In [111]:
plot_task_vector_variance_with_fit(task_vectors)

In [112]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [114]:
plot_pairwise_task_vector_variance(task_vectors)

In [113]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [109]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [115]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [3455.875  105.469   49.272   47.531   44.324   37.312   35.816   32.149
    0.621    0.05     0.012    0.01     0.006    0.005    0.003    0.003
    0.002    0.002    0.001    0.001    0.001    0.001    0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.      -0.      -0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
    0.       0.       0.       0.       0.       0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.       0.       0.       0.
    0.       0.       0.       0.       0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.       0.       0.       0.       0.  

In [120]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**6
model, log = train(config)

train_aef1b6d75d7d90d6dfd4bca8ceed7888 already completed
Loaded model from ../results/linear/train_aef1b6d75d7d90d6dfd4bca8ceed7888/checkpoint.pt


### 2**6 

In [121]:
model, train_task = load_model_and_task("train_aef1b6d75d7d90d6dfd4bca8ceed7888")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/64 [00:00<?, ?it/s]

In [122]:
plot_task_vector_variance_with_fit(task_vectors)

In [123]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [124]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [125]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [126]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [127]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/64 [00:00<?, ?it/s]